In [1]:
from datetime import date, datetime, timedelta

from data_engineering.bts.read_bts_ontime_to_flights import get_bts_flights_for_day
from data_engineering.adsbx.get_flights import get_adsbx_flights_for_day
from flights_ai_model.validation import flights_df_comparison
from data_engineering.adsbx.readsb import read_adsbx_readsb
from flights.flights_match_adsb import get_matching_icaos_in_flights
import polars as pl
DAY_MARGIN = timedelta(minutes=60)

test_date = date(2026,3,1)
df_adsb = read_adsbx_readsb()
df_gold = get_bts_flights_for_day(test_date)
df_gold = get_matching_icaos_in_flights(df_gold, df_adsb)
# df_test = get_adsbx_flights_for_day(test_date)
df_test = pl.read_parquet('/Volumes/T2-SSD/planequery/data/flights/trad-model/2026-03-01.parquet')

expr = (pl.col("landing_time").dt.date() == test_date) & (pl.col("takeoff_time").dt.date() == test_date)
min_time = datetime(test_date.year, test_date.month, test_date.day) + DAY_MARGIN
max_time = datetime(test_date.year, test_date.month, test_date.day) + timedelta(days=1) - DAY_MARGIN
expr2 = (pl.col("takeoff_time") > min_time) & (pl.col("landing_time") < max_time)
expr = expr
df_gold = df_gold.filter(expr)
df_test = df_test.filter(expr)
df_test = df_test.filter(pl.col("icao").is_in(df_gold.get_column("icao")))
start_time = datetime(test_date.year, test_date.month, test_date.day)
end_time = start_time + timedelta(days=1)
grace_period = timedelta(minutes=15)
df,stats = flights_df_comparison(df_test, df_gold, start_time=start_time, end_time=end_time, start_grace_period=grace_period, end_grace_period=grace_period) 

6 overlapping flights found for 2026-03-01

--- Match Summary (ICAO-based, ±30 min window) ---
Common ICAOs: 1817
Predicted flights: 7131  |  Gold flights: 6954
Correctly matched: 6942  |  Predicted unmatched: 189  |  Gold flights missed: 12
Precision: 97.35%  |  Recall: 99.83%  |  F1: 98.57%

--- Airport Ident Agreement (on time-matched flights) ---
takeoff_airport_ident: 6926/6927 matched (99.99%)
landing_airport_ident: 6916/6917 matched (99.99%)
overall: 13842/13844 matched (99.99%)


/var/folders/vc/71y607gj39b8jtlhqbbmxv000000gn/T/ipykernel_12540/2658501902.py:25: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df_test = df_test.filter(pl.col("icao").is_in(df_gold.get_column("icao")))


In [2]:
COLUMNS = ["icao", "takeoff_time", "takeoff_airport_ident", "landing_time", "landing_airport_ident", "takeoff_time_df2", "takeoff_airport_ident_df2", "landing_time_df2","landing_airport_ident_df2", "match_status"]
df = df.select(COLUMNS)

In [3]:
icaos_no_match = df.filter(pl.col("match_status") != "matched").get_column("icao").unique().to_list()
icaos_no_match = df.filter(pl.col("match_status") == "df2_only").get_column("icao").unique().to_list()

In [4]:
icaos_no_match

['ac9113',
 'ad504d',
 'a8e9fd',
 'ac77d3',
 'a6b435',
 'a7f63d',
 'a6ce36',
 'a2b8b8',
 'ac4bdd',
 'a826f1']

In [6]:
icao = icaos_no_match[4]
icao = "ac9113"
display(df_test.filter(pl.col("icao") == icao))
display(df_gold.filter(pl.col("icao") == icao))
display(df.filter(pl.col("icao") == icao).sort("takeoff_time"))

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""ac9113""","""""","""""",2026-03-01 01:39:49.602,"""KGJT""",2026-03-01 02:29:14.001,"""KDEN""",false,false,false,false,"""""","""""","""""",""""""
"""ac9113""","""""","""""",2026-03-01 03:41:54.372,"""KDEN""",2026-03-01 04:46:58.455,"""KLNK""",false,false,false,false,"""""","""""","""""",""""""
"""ac9113""","""""","""""",2026-03-01 15:41:28.427,"""KLNK""",2026-03-01 16:55:05.586,"""KDEN""",false,false,false,false,"""""","""""","""""",""""""
"""ac9113""","""""","""""",2026-03-01 18:49:07.052,"""KDEN""",2026-03-01 20:18:58.296,"""KSGU""",false,false,false,false,"""""","""""","""""",""""""
"""ac9113""","""""","""""",2026-03-01 21:25:45.577,"""KSGU""",2026-03-01 22:42:13.243,"""KDEN""",false,false,false,false,"""""","""""","""""",""""""


icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""ac9113""",null,"""N909EV""",2026-03-01 00:02:00,"""KDEN""",2026-03-01 00:40:00,"""KGJT""",false,false,false,false,null,null,null,"""A3"""
"""ac9113""",null,"""N909EV""",2026-03-01 01:45:00,"""KGJT""",2026-03-01 02:29:00,"""KDEN""",false,false,false,false,null,null,null,"""A3"""
"""ac9113""",null,"""N909EV""",2026-03-01 03:48:00,"""KDEN""",2026-03-01 04:46:00,"""KLNK""",false,false,false,false,null,null,null,"""A3"""
"""ac9113""",null,"""N909EV""",2026-03-01 15:41:00,"""KLNK""",2026-03-01 16:54:00,"""KDEN""",false,false,false,false,null,null,null,"""A3"""
"""ac9113""",null,"""N909EV""",2026-03-01 18:55:00,"""KDEN""",2026-03-01 20:19:00,"""KSGU""",false,false,false,false,null,null,null,"""A3"""
"""ac9113""",null,"""N909EV""",2026-03-01 21:29:00,"""KSGU""",2026-03-01 22:41:00,"""KDEN""",false,false,false,false,null,null,null,"""A3"""


icao,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,takeoff_time_df2,takeoff_airport_ident_df2,landing_time_df2,landing_airport_ident_df2,match_status
str,datetime[ms],str,datetime[ms],str,datetime[ms],str,datetime[ms],str,str
"""ac9113""",null,null,null,null,2026-03-01 00:02:00,"""KDEN""",2026-03-01 00:40:00,"""KGJT""","""df2_only"""
"""ac9113""",2026-03-01 01:39:49.602,"""KGJT""",2026-03-01 02:29:14.001,"""KDEN""",2026-03-01 01:45:00,"""KGJT""",2026-03-01 02:29:00,"""KDEN""","""matched"""
"""ac9113""",2026-03-01 03:41:54.372,"""KDEN""",2026-03-01 04:46:58.455,"""KLNK""",2026-03-01 03:48:00,"""KDEN""",2026-03-01 04:46:00,"""KLNK""","""matched"""
"""ac9113""",2026-03-01 15:41:28.427,"""KLNK""",2026-03-01 16:55:05.586,"""KDEN""",2026-03-01 15:41:00,"""KLNK""",2026-03-01 16:54:00,"""KDEN""","""matched"""
"""ac9113""",2026-03-01 18:49:07.052,"""KDEN""",2026-03-01 20:18:58.296,"""KSGU""",2026-03-01 18:55:00,"""KDEN""",2026-03-01 20:19:00,"""KSGU""","""matched"""
"""ac9113""",2026-03-01 21:25:45.577,"""KSGU""",2026-03-01 22:42:13.243,"""KDEN""",2026-03-01 21:29:00,"""KSGU""",2026-03-01 22:41:00,"""KDEN""","""matched"""


# a51b93 investigation

In [7]:
df = get_bts_flights_for_day(test_date).filter(pl.col("icao") == "aa35e1")
df

6 overlapping flights found for 2026-03-01


icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""aa35e1""",null,"""N757UW""",2026-03-01 21:06:00,"""KDFW""",2026-03-01 22:54:00,"""KMSN""",false,false,false,false,null,null,null,"""A3"""
"""aa35e1""",null,"""N757UW""",2026-03-01 23:52:00,"""KMSN""",2026-03-02 03:07:00,"""KPHX""",false,false,false,false,null,null,null,"""A3"""


In [28]:
df.filter(pl.col("icao") == "a1e6b9").sort("takeoff_time")

icao,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,takeoff_time_df2,takeoff_airport_ident_df2,landing_time_df2,landing_airport_ident_df2,match_status
str,datetime[ms],str,datetime[ms],str,datetime[ms],str,datetime[ms],str,str
"""a1e6b9""",null,null,null,null,2026-03-01 01:03:00,"""KELP""",2026-03-01 02:32:00,"""KHOU""","""df2_only"""
"""a1e6b9""",2026-03-01 03:46:12.314,"""KHOU""",2026-03-01 04:20:43.771,"""KCRP""",2026-03-01 03:46:00,"""KHOU""",2026-03-01 04:20:00,"""KCRP""","""matched"""
"""a1e6b9""",2026-03-01 11:49:28.642,"""KCRP""",2026-03-01 12:23:43.988,"""KHOU""",2026-03-01 11:49:00,"""KCRP""",2026-03-01 12:23:00,"""KHOU""","""matched"""
"""a1e6b9""",2026-03-01 13:37:11.616,"""KHOU""",2026-03-01 14:21:27.603,"""KDAL""",2026-03-01 13:37:00,"""KHOU""",2026-03-01 14:19:00,"""KDAL""","""matched"""
"""a1e6b9""",2026-03-01 15:21:28.277,"""KDAL""",2026-03-01 16:41:12.926,"""KSTL""",2026-03-01 15:21:00,"""KDAL""",2026-03-01 16:41:00,"""KSTL""","""matched"""
"""a1e6b9""",2026-03-01 18:04:55.807,"""KSTL""",2026-03-01 20:00:13.737,"""KLGA""",2026-03-01 18:05:00,"""KSTL""",2026-03-01 20:00:00,"""KLGA""","""matched"""


In [26]:
df.filter(pl.col("icao") == "a0b59d").sort("takeoff_time")

icao,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,takeoff_time_df2,takeoff_airport_ident_df2,landing_time_df2,landing_airport_ident_df2,match_status
str,datetime[ms],str,datetime[ms],str,datetime[ms],str,datetime[ms],str,str
"""a0b59d""",2026-03-01 16:08:36.994,"""KSLC""",2026-03-01 17:48:11.812,"""KPDX""",2026-03-01 16:13:00,"""KSLC""",2026-03-01 17:48:00,"""KPDX""","""matched"""
"""a0b59d""",2026-03-01 18:47:43.245,"""KPDX""",2026-03-01 20:10:58.403,"""KSLC""",2026-03-01 18:47:00,"""KPDX""",2026-03-01 20:10:00,"""KSLC""","""matched"""


In [8]:
df_gold.filter(pl.col("icao") == "a19982")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a19982""",null,"""N202AS""",2026-03-01 01:51:00,"""KPDX""",2026-03-01 07:46:00,"""PHNL""",false,false,false,false,null,null,null,"""A3"""
